In [ ]:
from reporting import platform_counts, fit_record, evaluation_counts, print_manuscript_summary
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import netCDF4 as nc
import xarray as xr
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import matplotlib.dates as mdates

import sys
sys.path.append("functions") 
from functions import latte, axis_ratio_from_Q, tangential_velocity, out_core_param_fit, model_uv_at_xy, vector_R2


### Drifter Data

In [ ]:
drifter_path = 'data/drifter_data.nc'

drifter_dataset = nc.Dataset(drifter_path, mode='r')

u_drif, v_drif = drifter_dataset['ve'][:].data, drifter_dataset['vn'][:].data
lon_drif, lat_drif = drifter_dataset['longitude'][:].data, drifter_dataset['latitude'][:].data
ID_drif =  drifter_dataset['ID'][:]
secs = drifter_dataset['time'][:].data
tim_drif = [datetime.utcfromtimestamp(t).strftime('%Y-%m-%d %H:%M:%S') for t in secs]
df_drifters = pd.DataFrame({'ID': ID_drif.astype(int), 'Lon': lon_drif, 'Lat': lat_drif, 'u': u_drif, 'v': v_drif, 'time': tim_drif})
df_drifters['time'] = pd.to_datetime(df_drifters['time'])
df_drifters = df_drifters[np.hypot(df_drifters.u, df_drifters.v) < 100]
df_drifters = df_drifters.reset_index().drop(columns=['index'])
df_drifters = df_drifters[df_drifters.time < pd.Timestamp('2024-02-10 00:00:00')]

theta_mean, phi_mean = np.float64(154.3), np.float64(-37.9)
df_drifters['x'] = (df_drifters.Lon - theta_mean) * 111000 * np.cos(np.deg2rad(phi_mean))
df_drifters['y'] = (df_drifters.Lat - phi_mean) * 111000

df_drifters = df_drifters[df_drifters.time <= np.datetime64('2024-01-27T06:00:00.000000000')]

df_drifters.head()


In [ ]:
# Find inner-core ESP parameters from just the drifter data
start = df_drifters.time.min() + pd.Timedelta(days=1)
end   = df_drifters.time.max() - pd.Timedelta(days=1)
times = pd.date_range(start, end, freq="D")

T  = df_drifters["time"].to_numpy()
XY = df_drifters[["x","y"]].to_numpy()
UV = df_drifters[["u","v"]].to_numpy()
ID = df_drifters["ID"].to_numpy()

results = []
dic_drif = {}

for t in times:
    m = (T >= (t - pd.Timedelta(days=1))) & (T <= (t + pd.Timedelta(days=1)))
    xi, yi = XY[m].T
    ui, vi = UV[m].T

    dic_drif[t] = pd.DataFrame(
        {"xi": xi, "yi": yi, "ui": ui, "vi": vi, "ID": ID[m]}
    )

    xc, yc, w, Q, Omega, r2_latte = latte(xi, yi, ui, vi)
    results.append({"xc": xc, "yc": yc, "w": w, "Q": Q, "Omega": Omega, "r2_latte": r2_latte, "time": t})

df_drif_res = pd.DataFrame(results)
df_drif_res["alpha"] = [axis_ratio_from_Q(Q) for Q in df_drif_res["Q"].to_list()]
df_drif_res


### Satellite Data

In [ ]:
ssh_path = 'data/satellite_data.nc'
ds_nc = nc.Dataset(ssh_path)

lon, lat = ds_nc['LONGITUDE'][:].data, ds_nc['LATITUDE'][:].data
u, v = ds_nc['UCUR'][:].data, ds_nc['VCUR'][:].data

t = pd.Timestamp("1985-01-01 00:00:00", tz="UTC") + pd.to_timedelta(ds_nc['TIME'][:].data, "D")
t = pd.DatetimeIndex(t).tz_convert(None)

u, v = np.where(np.abs(u) < 4.9, u, np.nan), np.where(np.abs(v) < 4.9, v, np.nan)

ds = xr.Dataset(
    {"u": (("time","lat","lon"), u), "v": (("time","lat","lon"), v)},
    coords={"time": t, "lat": lat, "lon": lon},
)

theta_mean, phi_mean = np.float64(154.3), np.float64(-37.9)
lon2d, lat2d = np.meshgrid(lon, lat)
ds = ds.assign_coords(
    x=(("lat","lon"), (lon2d-theta_mean)*111000*np.cos(np.deg2rad(phi_mean))),
    y=(("lat","lon"), (lat2d-phi_mean)*111000),
)

t0, t1 = ds.time.to_index()[0], ds.time.to_index()[-1]
start = t0.normalize() + pd.Timedelta(hours=6) + pd.Timedelta(days=int(t0 > t0.normalize() + pd.Timedelta(hours=6)))
end   = t1.normalize() + pd.Timedelta(hours=6) - pd.Timedelta(days=int(t1 < t1.normalize() + pd.Timedelta(hours=6)))
t6am = pd.date_range(start, end, freq="1D")

ds_sat = ds.interp(time=t6am)

# Find land mask
land_mask = np.isnan(ds.u.isel(time=0)) & np.isnan(ds.v.isel(time=0))  # (lat, lon)
ds = ds.assign(land_mask=land_mask)
ds_sat = ds_sat.assign(land_mask=land_mask)

# # Time slice
ds_sat = ds_sat.sel(time=ds_sat.time <= np.datetime64('2024-01-27T06:00:00.000000000'))

ds_sat


## Multi-platform LATTE application

#### Calculate inner-core ESP parameters

In [ ]:
initial_center = (0.0, 0.0)
xc_pre, yc_pre = initial_center
inner_count_rows = []
tracking_stop = "input record exhausted"
q11, q12, q22 = 1.0, 0.0, 1.0

rows = []
xyuv_rows = []

X = ds_sat.x.values
Y = ds_sat.y.values
xg = X.ravel()
yg = Y.ravel()

T_drif = df_drifters["time"].to_numpy()
rho_core = 35_000.0
dt = pd.Timedelta(days=1)  # ±1 day

flag = True
for time in ds_sat.time.values:
    ut = ds_sat.u.sel(time=time).values.ravel()
    vt = ds_sat.v.sel(time=time).values.ravel()

    m = (T_drif >= (time - dt)) & (T_drif <= (time + dt))
    dfm = df_drifters.iloc[np.flatnonzero(m)]
    xd = dfm["x"].to_numpy()
    yd = dfm["y"].to_numpy()
    ud = dfm["u"].to_numpy()
    vd = dfm["v"].to_numpy()
    
    xi = np.concatenate([xd, xg])
    yi = np.concatenate([yd, yg])
    ui = np.concatenate([ud, ut])
    vi = np.concatenate([vd, vt])

    platform_labels = np.concatenate([np.full(len(xd), "drifter"), np.full(len(xg), "sat")])

    ########## Approxiamte inner-core data from previous day ############################
    dx = xi - xc_pre
    dy = yi - yc_pre
    rho2 = q11*dx*dx + 2*q12*dx*dy + q22*dy*dy
    core = rho2 <= rho_core**2
    initial_counts = platform_counts(platform_labels, core, xi, yi, ui, vi)
    xc, yc, w, Q, Omega, r2_latte = latte(xi[core], yi[core], ui[core], vi[core])
    ###############################################################################

    ########## Reapproximate inner-core parameters with more accurate core data #############
    xc_pre, yc_pre = xc, yc
    dx = xi - xc_pre
    dy = yi - yc_pre
    rho2 = q11*dx*dx + 2*q12*dx*dy + q22*dy*dy
    core = rho2 <= rho_core**2
    xc, yc, w, Q, Omega, r2_latte = latte(xi[core], yi[core], ui[core], vi[core])
    ###############################################################################

    xyuv_rows.append(pd.DataFrame({
        "time": time,
        "core": "inner",
        "xi": xi[core], "yi": yi[core], "ui": ui[core], "vi": vi[core]
    }))

    if np.isfinite(xc) and np.isfinite(yc) and (np.hypot(xc-xc_pre, yc-yc_pre) < 100_000) and (w < 0):
        refined_counts = platform_counts(platform_labels, core, xi, yi, ui, vi)
        inner_count_rows.append({"time": time,
                                 **{"initial_" + k: v for k, v in initial_counts.items()},
                                 **{"refined_" + k: v for k, v in refined_counts.items()}})
        rows.append(dict(xc=xc, yc=yc, w=w, Q=Q, Omega=Omega, r2=r2_latte, time=time))
        xc_pre, yc_pre = xc, yc
        q11, q12, q22 = Q[0,0], Q[0,1], Q[1,1]   # update deformation metric for next step
    else:
        tracking_stop = f"rejected {time}: finite centre={np.isfinite(xc) and np.isfinite(yc)}, refinement step={np.hypot(xc-xc_pre, yc-yc_pre)} m, omega={w}"
        break
df_xyuv = pd.concat(xyuv_rows, ignore_index=True) if xyuv_rows else pd.DataFrame(
    columns=["time","core","xi","yi","ui","vi"]
)
df_multimodal_res = pd.DataFrame(rows, columns=["xc","yc","w","Q","Omega","time"])
df_multimodal_res["alpha"] = [axis_ratio_from_Q(Q) for Q in df_multimodal_res["Q"].to_list()]
df_multimodal_res


In [ ]:
# quick comparison between multi-paltform and drifter results
fig, axs = plt.subplots(1, 2, figsize=(10,4), sharey=False)
axs[0].plot(df_multimodal_res.xc/1e3, df_multimodal_res.yc/1e3, label='multi');
axs[0].plot(df_drif_res.xc/1e3, df_drif_res.yc/1e3, label='drifter'); axs[0].axis('equal')
axs[0].set_xlabel('x (km)'); axs[0].set_ylabel('y (km)'); axs[0].legend()

axs[1].plot(df_multimodal_res.alpha, df_multimodal_res.time, label=r'$\alpha$ multi')
axs[1].plot(df_drif_res.alpha, df_drif_res.time, label=r'$\alpha$ drifter')
axs[1].set_xlabel('aspect ratio'); axs[1].set_ylabel('Time')
axs[1].legend(); axs[1].axvline(1, lw=.5, color='k')
plt.tight_layout()


In [ ]:
# Inner-core mutli-platform data used 
cmap = plt.colormaps['tab20']
for t, time in enumerate(df_xyuv.time.unique()):
    df = df_xyuv[(df_xyuv.time == time) & (df_xyuv.core == 'inner')]
    plt.scatter(df.xi, df.yi, s=10, color=cmap(t % cmap.N))
plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.axis('equal')


#### Calculate outer-core ESP parameters

In [ ]:
rho_outer = 100_000.0
dic_outer_mask_data = {}

Rcs, psi0s, Omega_opts = [], [], []
xyuv_rows = []
outer_records = []
outer_count_rows = []

for k in range(len(df_multimodal_res)):
    data = df_multimodal_res.iloc[k]
    time = data.time  # or data.Time if that's your column

    ut = ds_sat.u.sel(time=time).values.ravel()
    vt = ds_sat.v.sel(time=time).values.ravel()

    m = (df_drifters["time"].to_numpy() >= (time - dt)) & (df_drifters["time"].to_numpy() <= (time + dt))
    dfm = df_drifters.iloc[np.flatnonzero(m)]
    xd, yd = dfm["x"].to_numpy(), dfm["y"].to_numpy()
    ud, vd = dfm["u"].to_numpy(), dfm["v"].to_numpy()

    xi = np.concatenate([xd, xg])
    yi = np.concatenate([yd, yg])
    ui = np.concatenate([ud, ut])
    vi = np.concatenate([vd, vt])

    xc, yc, Q, Omega = float(data.xc), float(data.yc), np.asarray(data.Q), float(data.Omega)
    q11, q12, q22 = Q[0,0], Q[0,1], Q[1,1]

    dx = xi - xc
    dy = yi - yc
    rho2 = q11*dx*dx + 2*q12*dx*dy + q22*dy*dy

    outer = rho2 <= rho_outer**2
    xi_o, yi_o, ui_o, vi_o = xi[outer], yi[outer], ui[outer], vi[outer]

    xyuv_rows.append(pd.DataFrame({
        "time": time, "core": "outer",
        "xi": xi_o, "yi": yi_o, "ui": ui_o, "vi": vi_o
    }))

    fit_details = {}
    Rc, psi0, Omega_opt = out_core_param_fit(xi_o, yi_o, ui_o, vi_o, xc, yc, Q, Omega0=Omega, plot=(k == 10), pred_flag=True, fit_details=fit_details)
    outer_records.append(fit_record(time, fit_details))
    platform_labels = np.concatenate([np.full(len(xd), "drifter"), np.full(len(xg), "sat")])
    candidate_counts = platform_counts(platform_labels, outer, xi, yi, ui, vi)
    used = np.zeros(len(xi), dtype=bool)
    used[np.flatnonzero(outer)[fit_details.get("observation_indices", np.array([], dtype=int))]] = True
    used_counts = platform_counts(platform_labels, used, xi, yi, ui, vi)
    outer_count_rows.append({"time": time,
                             **{"outer_candidates_" + k: v for k, v in candidate_counts.items()},
                             **{"outer_fit_" + k: v for k, v in used_counts.items()}})

    Rcs.append(Rc); psi0s.append(psi0); Omega_opts.append(Omega_opt)

df_xyuv_o = pd.concat(xyuv_rows, ignore_index=True) if xyuv_rows else pd.DataFrame(
    columns=["time","core","xi","yi","ui","vi"]
)

df_xyuv = pd.concat([df_xyuv, df_xyuv_o], ignore_index=True)
df_xyuv = (
    df_xyuv
    .sort_values(by="core", key=lambda s: s.eq("outer"))  # keep inner first
    .drop_duplicates(subset=["time", "xi", "yi", "ui", "vi"], keep="first")
    .reset_index(drop=True)
)

df_multimodal_res["Rc"] = Rcs
df_multimodal_res["psi0"] = psi0s
df_multimodal_res["Omega"] = Omega_opts
df_multimodal_res["w"] = [Omega*(Q[0,0] + Q[1,1]) for (Omega,Q) in zip(df_multimodal_res.Omega, df_multimodal_res.Q)]
df_multimodal_res


In [ ]:
# Find R2
R2_vals = []
for _, row in df_multimodal_res.iterrows():
    df = df_xyuv[df_xyuv.time == row.time]
    xi = df.xi.to_numpy(float)
    yi = df.yi.to_numpy(float)
    u  = df.ui.to_numpy(float)
    v  = df.vi.to_numpy(float)
    uhat, vhat = model_uv_at_xy(xi, yi, row.xc, row.yc, np.asarray(row.Q), row.Omega, row.Rc)
    R2_vals.append(vector_R2(u, v, uhat, vhat))
df_multimodal_res['R2'] = R2_vals
df_multimodal_res


In [ ]:
# Outer-core mutli-platform data used 
cmap = plt.colormaps['tab20']
for t, time in enumerate(df_xyuv.time.unique()):
    df = df_xyuv[(df_xyuv.time == time)]
    plt.scatter(df.xi, df.yi, s=10, color=cmap(t % cmap.N))

plt.axis('equal')


In [ ]:
# Reconstruct velocity field using ESP parameters
X = ds_sat.x.values
Y = ds_sat.y.values
nt = len(df_multimodal_res)

U = np.empty((nt, X.shape[0], X.shape[1]))
V = np.empty((nt, X.shape[0], X.shape[1]))

for k in range(nt):
    data = df_multimodal_res.iloc[k]

    dx = X - data.xc
    dy = Y - data.yc

    q11, q12, q22 = data.Q[0,0], data.Q[0,1], data.Q[1,1]

    rho2 = q11*dx**2 + 2*q12*dx*dy + q22*dy**2
    fac = data.Omega * np.exp(-rho2 / data.Rc**2)

    U[k] = -fac * (q12*dx + q22*dy)
    V[k] =  fac * (q11*dx + q12*dy)

mask = ds_sat.land_mask.values[None, :, :]   # (1, lat, lon)
U = np.where(mask, np.nan, U)
V = np.where(mask, np.nan, V)

ds_model = xr.Dataset(
    data_vars=dict(
        u=(("time","lat","lon"), U),
        v=(("time","lat","lon"), V),
    ),
    coords=dict(
        time=df_multimodal_res.time.values,
        lat=ds_sat.lat.values,
        lon=ds_sat.lon.values,
        x=(("lat","lon"), X),
        y=(("lat","lon"), Y),
    ),
)


In [ ]:
def plot_model(df_res, ds_sat, ds_model, df_xyuv, df_drifters, time=None, spac=1, scale=20,
                xlim=(-410, 200), ylim=(-160, 260), plot_data=False):

    if time is None:
        time = ds_model.time.values[0]

    sat = ds_sat.sel(time=time, method="nearest")
    mod = ds_model.sel(time=time, method="nearest")
    data = df_res[df_res.time == time].iloc[0]

    X = sat.x.values / 1e3
    Y = sat.y.values / 1e3

    us, vs = sat.u.values, sat.v.values
    um, vm = mod.u.values, mod.v.values

    sp_s = np.hypot(us, vs)
    sp_m = np.hypot(um, vm)

    dx = sat.x.values - data.xc
    dy = sat.y.values - data.yc
    q11, q12, q22 = data.Q[0,0], data.Q[0,1], data.Q[1,1]
    rho2_grid = q11*dx**2 + 2*q12*dx*dy + q22*dy**2

    roi = (X >= xlim[0]) & (X <= xlim[1]) & (Y >= ylim[0]) & (Y <= ylim[1])
    sp_s_roi = np.where(roi, sp_s, np.nan)
    sp_m_roi = np.where(roi, sp_m, np.nan)
    roi_quiv = roi & (rho2_grid >= 100_000**2)

    vals = np.concatenate([sp_s_roi[roi], sp_m_roi[roi]])
    vmax = np.nanmax(vals)
    levels = np.arange(0, vmax + .05, .05)

    X_rho, Y_rho = np.meshgrid(np.linspace(xlim[0]*1e3, xlim[1]*1e3, 100),
                               np.linspace(ylim[0]*1e3, ylim[1]*1e3, 100))
    dx = X_rho - data.xc
    dy = Y_rho - data.yc
    rho2 = q11*dx**2 + 2*q12*dx*dy + q22*dy**2

    fig, axs = plt.subplots(1, 2, figsize=(15, 7), sharex=True, sharey=True, constrained_layout=True)

    # ----- coloured center path -----
    x_path = df_res.xc.to_numpy() / 1e3
    y_path = df_res.yc.to_numpy() / 1e3
    t_path = pd.to_datetime(df_res.time)
    t_num = mdates.date2num(t_path)

    pts = np.array([x_path, y_path]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)

    norm_t = Normalize(t_num.min(), t_num.max())
    lc0 = LineCollection(segs, cmap="plasma", norm=norm_t, linewidth=2, zorder=2)
    lc0.set_array(t_num[:-1])

    lc1 = LineCollection(segs, cmap="plasma", norm=norm_t, linewidth=2, zorder=2)
    lc1.set_array(t_num[:-1])

    # ---- Satellite ----
    h0 = axs[0].contourf(X, Y, sp_s_roi, levels=levels, cmap="Blues_r")

    axs[0].quiver(X[::spac,::spac], Y[::spac,::spac],
                  np.where(roi_quiv[::spac,::spac], us[::spac,::spac], np.nan),
                  np.where(roi_quiv[::spac,::spac], vs[::spac,::spac], np.nan),
                  scale=scale, width=.0015, alpha=.4)

    axs[0].scatter(data.xc/1e3, data.yc/1e3, marker="x", color="r", s=50, zorder=3)
    axs[0].add_collection(lc0)
    axs[0].contour(X_rho/1e3, Y_rho/1e3, rho2,
                   levels=[(data.Rc**2)/2], colors="r", linewidths=2,
                   zorder=3, linestyles='--')

    if plot_data:
        df = df_xyuv[(df_xyuv.time == time) & (df_xyuv.core == 'outer')]
        axs[0].quiver(df.xi/1e3, df.yi/1e3, df.ui, df.vi,
                      scale=scale, width=.0025, color='g',
                      label='Outer-core data', zorder=2)
        df = df_xyuv[(df_xyuv.time == time) & (df_xyuv.core == 'inner')]
        axs[0].quiver(df.xi/1e3, df.yi/1e3, df.ui, df.vi,
                      scale=scale, width=.0025, color='m',
                      label='Inner-core data', zorder=2)

    for i, d in enumerate(df_drifters.ID.unique()):
        df = df_drifters[df_drifters.ID == d]
        axs[0].plot(df.x/1e3, df.y/1e3, zorder=1, lw=1, alpha=.5,
                    color='tab:pink' if i == 0 else 'tab:orange')

    axs[0].set_aspect("equal")
    axs[0].legend(loc='lower left')

    # ---- Model ----
    h1 = axs[1].contourf(X, Y, sp_m_roi, levels=levels, cmap="Blues_r")
    axs[1].quiver(X[::spac,::spac], Y[::spac,::spac],
                  np.where(roi[::spac,::spac], um[::spac,::spac], np.nan),
                  np.where(roi[::spac,::spac], vm[::spac,::spac], np.nan),
                  scale=scale, alpha=.5, width=.0015)
    axs[1].scatter(data.xc/1e3, data.yc/1e3, marker="x", color="r", s=50,
                   label='Eddy center', zorder=3)
    axs[1].add_collection(lc1)
    axs[1].contour(X_rho/1e3, Y_rho/1e3, rho2,
                   levels=[(data.Rc**2)/2], colors="r", linewidths=2,
                   zorder=3, linestyles='--')
    axs[1].plot([np.nan, np.nan], [np.nan, np.nan], lw=2, color='r',
                label='Max tangential\nvelocity', ls='--')

    for i, d in enumerate(df_drifters.ID.unique()):
        df = df_drifters[df_drifters.ID == d]
        axs[1].plot(df.x/1e3, df.y/1e3, zorder=1, lw=1, alpha=.5,
                    color='tab:pink' if i == 0 else 'tab:orange',
                    label='Drifter 1' if i == 0 else 'Drifter 2')

    axs[1].set_aspect("equal")
    axs[1].legend(loc='lower left')

    gap = 20
    for ax in axs:
        ax.set_xlim(xlim[0] + gap, xlim[1] - gap)
        ax.set_ylim(ylim[0] + gap, ylim[1] - gap)
        ax.set_xlabel("x (km)")
    axs[0].set_ylabel("y (km)")

    axs[0].text(-0.05, 1.05, "a)", transform=axs[0].transAxes, fontsize=13,
                fontweight="bold", va="top", ha="left")
    axs[1].text(-0.05, 1.05, "b)", transform=axs[1].transAxes, fontsize=13,
                fontweight="bold", va="top", ha="left")

    # speed colourbar
    fig.colorbar(h1, ax=axs, label="Current speed (m s$^{-1}$)", fraction=0.0165, pad=0.02)

    # time colourbar
    cbar2 = fig.colorbar(lc1, ax=axs, fraction=0.0165, pad=0.025)
    cbar2.set_label("Center path time")
    cbar2.ax.yaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))

    plt.show()


In [ ]:
figure_time = df_multimodal_res.time.values[10]
plot_model(df_multimodal_res, ds_sat, ds_model, df_xyuv, df_drifters, time=figure_time, plot_data=True)


In [ ]:
# Results
df_paper = df_multimodal_res.copy()
df_paper = df_paper.drop(columns='Q')
df_paper.loc[np.arange(0, 100, 10)]


In [ ]:
# Time series
clrs = plt.colormaps['tab10']
fig, axs = plt.subplots(5, 1, figsize=(10,6), sharex=True)
lw = 2

axs[0].plot(df_multimodal_res.time, df_multimodal_res.w, color=clrs(0), lw=lw)
axs[0].set_ylabel(r'$\omega$ (s$^{-1}$)')

axs[1].plot(df_multimodal_res.time, df_multimodal_res.Omega, color=clrs(1), lw=lw)
axs[1].set_ylabel(r'$\Omega$ (s$^{-1}$)')

axs[2].plot(df_multimodal_res.time, df_multimodal_res.alpha, color=clrs(2), lw=lw)
axs[2].set_ylabel(r'$\alpha$')
axs[2].set_ylim(1, None)

axs[3].plot(df_multimodal_res.time, df_multimodal_res.Rc/1e3, color=clrs(3), lw=lw)
axs[3].set_ylabel(r'$R_c$ (km)')

axs[4].plot(df_multimodal_res.time, df_multimodal_res.R2, color=clrs(4), lw=lw)
axs[4].set_ylabel(r'$R^2$')
axs[4].set_xlabel('Time')
axs[4].set_ylim(0,1)

fig.subplots_adjust(hspace=0)


### Manuscript results summary
Computed from this run. All dates are UTC; final refined parameters are reported.
This notebook demonstrates ESP on real data; idealised experiments are outside its scope.

In [ ]:
summary_counts = pd.DataFrame(inner_count_rows).merge(pd.DataFrame(outer_count_rows), on="time", how="outer")
summary_counts = summary_counts.merge(evaluation_counts(df_xyuv), on="time", how="left")
print_manuscript_summary(
    "LATTE — drifter and satellite", df_multimodal_res,
    {"Drifter IDs": df_drifters.ID.unique().tolist(),
     "Selected drifter interval": f"{df_drifters.time.min()} to {df_drifters.time.max()} UTC",
     "Drifter observation count": len(df_drifters),
     "Satellite analysis interval": f"{ds_sat.time.values[0]} to {ds_sat.time.values[-1]} UTC",
     "Drifter aggregation half-window": dt,
     "Local origin (longitude, latitude)": (theta_mean, phi_mean),
     "Initial centre (m)": initial_center,
     "Initial/refined core radius (km)": rho_core / 1000,
     "Outer elliptical radius limit (km)": rho_outer / 1000,
     "Acceptance": "finite centre, refinement displacement <100 km, negative core vorticity",
     "Tracking termination": tracking_stop},
    counts=summary_counts, outer_records=outer_records, snapshot_time=figure_time,
    notes=["Both core selections use the carried-forward deformation metric, not a newly fitted Q between those two stages.",
           "The displacement check currently compares the final centre with the preliminary same-day centre, not the previous day's centre.",
           "Velocity R² uses the deduplicated union of inner and outer evaluation rows.",
           "Counts distinguish actual finite drifter and satellite contributions at each stage."],
    input_files=["LATTE_app2_multiplatform_data.ipynb", "data/drifter_data.nc", "data/satellite_data.nc"])
drifter_counts = pd.DataFrame([{"time": time, "drifter_pairs": int(np.isfinite(samples[["xi", "yi", "ui", "vi"]].to_numpy()).all(axis=1).sum()),
                               "drifter_ids": samples.ID.nunique()} for time, samples in dic_drif.items()])
print_manuscript_summary(
    "LATTE — drifter-only inner-core comparison", df_drif_res.assign(source="drifter"),
    {"Drifter aggregation half-window": pd.Timedelta(days=1),
     "Outer-core parameters": "not estimated; full-field velocity R² is unavailable"},
    counts=drifter_counts,
    notes=["R2_linear_core is the inner linear-flow fit; it is not the Gaussian reconstruction R².",
           "Rc, psi0 and R2_velocity are unavailable for the drifter-only comparison."])
